## PART III:  Error analysis Bootstrap method

In [ ]:
import sys,os
import numpy as np
import scipy
from scipy.integrate import quad
from scipy.stats import kde
from scipy.stats import chi2 as spchi2
from scipy.optimize import minimize,leastsq
import matplotlib
import matplotlib.pyplot as py
import matplotlib.gridspec as gridspec
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib import rc
rc('font',**{'family':'sans-serif','sans-serif':['Helvetica']})
rc('text', usetex=True)  
%matplotlib inline  
%config InlineBackend.figure_format = 'retina'
from tools import tex,save, load,fill_between,plot_hist,ProgressBar
from tools import EVENTS,VEC4
import numpy.linalg as LA 
from scipy.interpolate import interp1d
from scipy.optimize import fsolve,brentq,newton,fmin
from numpy.random import randn

# The signal + background

Get data

In [ ]:
path='samples/sb.lhe.gz'
sb=EVENTS(path)
sb.D['m']=[]
for i in range(sb.nevents):
    particles=sb.EVENTS[i]
    for p in particles:
        if p['pid']==-13: mub=p['mom']
        elif p['pid']==13: mu=p['mom']
    z=mu+mub
    sb.D['m'].append((z*z)**0.5) 

construct observable

In [ ]:
lower_binning = 40 #40
upper_binning = 130 #200
number_of_bins = 100 #100
N,E=np.histogram(sb.D['m'],bins=number_of_bins,range=(lower_binning,upper_binning))

M=0.5*(E[:-1]+E[1:])
NTOT = np.sum(N)
SNSQ = np.sum(np.sqrt(N))
OBS  = N/float(NTOT)
ERR  = np.zeros(OBS.size)
for k in range(len(ERR)):
        ERR[k] = np.sqrt(N[k])/float(NTOT)

ax=py.subplot(111)
ax.errorbar(M,OBS,yerr=ERR,fmt='g.')
ax.set_ylabel(tex(r'sig+bkg~(a.u)'),size=20)
ax.set_xlabel(tex(r'm (GeV)'),size=20)
ax.semilogy();

Define model

In [ ]:
rs = 1300.0
model_sig = lambda p,m: p[0]*m**2 /((m**2-p[1]**2)**2+m**2 *p[2]**2)*(1+p[3]*(m/rs) + p[4]*(m/rs)**2)
model_bkg = lambda p,m: p[5]/m**(p[6])
model_sb  = lambda p,m: model_sig(p,m) + model_bkg(p,m)

Perform single fit 

In [ ]:
# single fit
params_input = [1,91,1,1,1,1,1]
res  = lambda p,OBS: (OBS-model_sb(p,M))/ERR
p0   = leastsq(res,params_input,args=(OBS),full_output=1)[0]
res0 = res(p0,OBS)
chi2sb  = np.dot(res0,res0)
chi2dof = chi2sb/(OBS.size-p0.size)
for i in range(p0.size): print 'p%d = %f'%(i,p0[i])
print 'chi^2/dof=',chi2dof
ax = py.subplot(111)
ax.errorbar(M,OBS,yerr=ERR,fmt='g.')

Mmin = np.amin(M)
Mmax = np.amax(M)
M_   = np.linspace(Mmin,Mmax,1000)
ax.plot(M_,model_sb(p0,M_),'b-')
ax.set_ylabel(tex(r'sig+bkg~(a.u)'),size=20)
ax.set_xlabel(tex(r'm (GeV)'),size=20)
ax.semilogy();

MC fits

In [ ]:
# MC fit
nrep = 100
bar  = ProgressBar(nrep,'computing MC fits ')
POPT = []
REP  = []
CHI2 = []
CHI2DOF = []
for i in range(nrep):
    bar.animate(i+1)
    OBSk = OBS + randn(OBS.size)*ERR
    fit  = leastsq(res,p0,args=(OBSk),full_output=1)
    resk = res(fit[0],OBSk)
    CHI2.append(np.dot(resk,resk))
    CHI2DOF.append(np.dot(resk,resk)/(OBS.size-p0.size))
    POPT.append(fit[0])
    REP.append(OBSk)

In [ ]:
nbins = 50
fig = py.figure(1, figsize=(35,5))

ax  = py.subplot(151)
N,E = np.histogram(CHI2,bins=nbins)
plot_hist(ax,'r',E,N,symbol='-')
ax.set_ylabel(tex(r'N_i'),size=20)
ax.set_xlabel(tex(r'\chi^2'),size=20)

bx  = py.subplot(152)
Nb,Eb = np.histogram(CHI2DOF,bins=nbins)
plot_hist(bx,'r',Eb,Nb,symbol='-')
bx.set_ylabel(tex(r'N_i'),size=20)
bx.set_xlabel(tex(r'\chi^2/dof'),size=20);

In [ ]:
fig = py.figure(1, figsize=(11,6))

t = np.arange(0,N.size, 1)
A = []
for i in t:
    ll = (E[i]+E[i+1])/2 
    A.append(ll) 
    
values1 = py.hist(np.random.noncentral_chisquare(OBS.size-p0.size, chi2sb, nrep),bins=nbins, normed=False,color='#eeefff')
values3 = py.plot(A,N,'r-', linewidth=2.0)
values2 = py.plot(A,N,'ro',markersize=8)

py.show()

MC parameters 

In [ ]:
fig = py.figure(1, figsize=(12,4))
ax  = py.subplot(121)
N,E = np.histogram([popt[1] for popt in POPT])

t = np.arange(0,N.size, 1)
A = []
for i in t:
    ll = (E[i]+E[i+1])/2 
    A.append(ll) 
    
mu    = np.mean(E)
sigma = np.std(E)
print [mu, sigma]

plot_hist(ax,'r',E,N,symbol='-') 
ax.set_xlabel(tex('M_Z'),size=20)

ax  = py.subplot(122)
N,E = np.histogram([popt[2] for popt in POPT])
plot_hist(ax,'r',E,N,symbol='-')
ax.set_xlabel(tex(r'$\Gamma_Z$'),size=20)


t = np.arange(0,N.size, 1)
A = []
for i in t:
    ll = (E[i]+E[i+1])/2 
    A.append(ll) 
    
mu    = np.mean(E)
sigma = np.std(E)
print [mu, sigma]

plot results

In [ ]:
M0   = np.linspace(40,130,200)
OBSK = np.zeros((len(POPT),len(M0)))
for k in range(nrep): OBSK[k]=model_sb(POPT[k],M0)
OBS0   = model_sb(p0,M0)
std    = np.std(OBSK,axis=0)
OBSMAX = OBS0 + std
OBSMIN = OBS0 - std

I=[i for i in range(OBSMIN.size) if OBSMIN[i]>0]
M0 = np.array([M0[i] for i in I])
OBSMAX = np.array([OBSMAX[i] for i in I])
OBSMIN = np.array([OBSMIN[i] for i in I])
OBS0 = np.array([OBS0[i] for i in I])




py.figure(figsize=(5*3,4*1))

ax = py.subplot(131)
fill_between(M0,y1=OBSMIN,y2=OBSMAX, ax=ax, facecolor='b',alpha=0.4,label='fit~CL68\%')
#ax.plot(M0,OBSMIN)
#ax.plot(M0,OBSMAX)


ax.errorbar(M,OBS,yerr=ERR,fmt='r.',label=tex('sig+bkg~data'))
ax.semilogy()
ax.legend(fontsize=15,frameon=0,loc=3)
ax.set_ylabel(tex(r'sig+bkg~(a.u)'),size=20)
ax.set_xlabel(tex(r'm (GeV)'),size=20)

ax = py.subplot(132)
fill_between(M0,y1=OBSMIN/OBS0,y2=OBSMAX/OBS0, ax=ax, facecolor='b',alpha=0.4,label='fit~CL68\%')
ax.errorbar(M,OBS/model_sb(p0,M),yerr=ERR/model_sb(p0,M),fmt='k.')
ax.set_ylabel(tex(r'Data/Fit'),size=20)
ax.set_xlabel(tex(r'm (GeV)'),size=20)

ax = py.subplot(133)
fill_between(M0,y1=OBSMIN/OBS0,y2=OBSMAX/OBS0, ax=ax, facecolor='b',alpha=0.4,label='fit~CL68\%')
ax.errorbar(M,OBS/model_sb(p0,M),yerr=ERR/model_sb(p0,M),fmt='k.')
ax.set_xlim(40,120)
ax.set_ylim(0.5,1.5);
ax.set_ylabel(tex(r'Data/Fit'),size=20)
ax.set_xlabel(tex(r'm (GeV)'),size=20);